# Kết quả thu thập và chất lượng corpus

## Kết quả bộ MS1 đã chốt

Các số liệu sau tổng hợp toàn bộ năm dải được giao. Mỗi paper chỉ xuất hiện một lần. Số version được cộng từ chi tiết từng paper; không dùng một trường tổng không khớp chi tiết để thay số version của cả dải.

In [ ]:
from pathlib import Path
import sys,json,hashlib
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,HTML,Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'_config.yml').exists())
sys.path.insert(0,str(ROOT/'utils'))
from preliminary_results import load_members,corpus_metrics,session_members,run_range,paper_timings
folder=ROOT/'summaries/milestone1/frozen-20261127'
receipt=json.loads((folder/'provenance.json').read_text(encoding='utf-8'))
raw=(folder/'corpus.json').read_bytes()
assert hashlib.sha256(raw).hexdigest()==receipt['public_sha256']
papers=json.loads(raw)['papers']
assert len(papers)==25000 and len({p['paper_id'] for p in papers})==25000
members=load_members(ROOT)
sessions=session_members(ROOT,members)
measurement=corpus_metrics(members)[-1]
def table(rows):
    frame=pd.DataFrame(rows,dtype=object).where(lambda f:f.notna(),'Không áp dụng')
    def fmt(v):
        if isinstance(v,int) and not isinstance(v,bool):return f'{v:,}'.replace(',','.')
        if isinstance(v,float):return f'{v:,.2f}'.replace(',','_').replace('.',',').replace('_','.')
        return str(v)
    display(HTML(frame.map(fmt).to_html(index=False,escape=True,border=0)))


In [ ]:
n=len(papers); passed=sum(p['valid'] is True for p in papers)
versions=sum(p['discovery_versions'] for p in papers)
refs=sum(p['counts']['references'] for p in papers)
source=sum(p['counts']['source_versions'] for p in papers)
no_source=sum(p['counts']['no_source_versions'] for p in papers)
table([{'Chỉ số toàn nhóm':'Paper/ID đã xử lý','Giá trị':n},
       {'Chỉ số toàn nhóm':'Discovery thành công','Giá trị':sum(p['discovery_status']=='success' for p in papers)},
       {'Chỉ số toàn nhóm':'Version xác định được','Giá trị':versions},
       {'Chỉ số toàn nhóm':'Version có source theo kiểm định','Giá trị':source},
       {'Chỉ số toàn nhóm':'Version no_source theo kiểm định','Giá trị':no_source},
       {'Chỉ số toàn nhóm':'References đã ghi nhận','Giá trị':refs},
       {'Chỉ số toàn nhóm':'References đã ghi nhận / paper được giao','Giá trị':refs/n}])
table([{'Kết quả kiểm định':'Đạt','Paper':passed,'Tỷ lệ (%)':100*passed/n},
       {'Kết quả kiểm định':'Không đạt toàn bộ quy tắc','Paper':n-passed,'Tỷ lệ (%)':100*(n-passed)/n}])
fig,ax=plt.subplots(figsize=(7,3))
ax.barh(['Bộ corpus MS1'],[passed],label='Đạt kiểm định')
ax.barh(['Bộ corpus MS1'],[n-passed],left=[passed],label='Không đạt toàn bộ quy tắc')
ax.set_xlabel('Paper');ax.legend();plt.tight_layout();plt.show()

Tỷ lệ đạt kiểm định là tỷ lệ paper đáp ứng toàn bộ hợp đồng tại thời điểm chốt. Record no_source hợp lệ vẫn có thể đạt kiểm định. Ngược lại, references hoặc metadata không hợp lệ có thể làm paper không đạt dù source đã hiện diện. Những trường hợp này không được loại khỏi mẫu số 25.000 paper.

## Nhóm trạng thái chất lượng

Mỗi paper chỉ được đếm một lần trong từng nhóm; một paper có thể thuộc nhiều nhóm. Bảng mô tả kết quả kiểm tra của toàn corpus, không phải danh sách công việc đang chờ từng thành viên thực hiện.

In [ ]:
counts=Counter(code for p in papers if not p['valid'] for code in set(p['error_codes']))
labels={'upstream_failure':'Bước thu thập/xử lý không đáp ứng hợp đồng','artifact_location':'Đường dẫn artifact không khớp',
'invalid_corpus_file':'Tệp corpus không đáp ứng quy tắc','invalid_reference':'Reference không đáp ứng schema',
'invalid_source':'Source không đáp ứng quy tắc'}
table([{'Nhóm kết quả':labels.get(k,k),'Paper ghi nhận':v} for k,v in counts.most_common()])

## Thực nghiệm kích thước trước/sau lọc

Phép đo này sử dụng tập paper có đủ sidecar trước/sau lọc trong các bản cumulative đã lưu. Đây là thực nghiệm dung lượng trên cùng tập ghép cặp, tách khỏi số đếm corpus cuối. Đơn vị trước/sau đều là bytes/paper sau khi cộng các version của paper; trung bình gộp theo trọng số số paper ghép cặp.

In [ ]:
table([{'Phạm vi':'Tập ghép cặp của cả nhóm','Paper có cặp đo':measurement['paired'],
        'Trước lọc (bytes/paper)':measurement['mean_before'],'Sau lọc (bytes/paper)':measurement['mean_after'],
        'Giảm dung lượng (%)':100*(1-measurement['mean_after']/measurement['mean_before'])}])

## Thực nghiệm metadata tài liệu tham khảo

Các bản cumulative dùng cho phép đo ghi riêng paper có kết quả references đọc được, gồm danh sách rỗng hợp lệ. Tỷ lệ truy vấn đọc được và tỷ lệ reference đủ metadata có mẫu số khác nhau. Đủ metadata yêu cầu ref_arxiv_id, ref_title, ref_authors, ref_submission_date và ref_s2_id có giá trị. Các chỉ số sau thuộc phép đo đã lưu, không dùng mẫu số references corpus cuối cho tử số từ lần đo trước.

In [ ]:
table([{'Chỉ số thực nghiệm':'Paper có kết quả references đọc được','Tử số':measurement['retrieved'],
        'Mẫu số':measurement['discovered'],'Tỷ lệ (%)':100*measurement['retrieved']/measurement['discovered']},
       {'Chỉ số thực nghiệm':'References đủ năm trường metadata','Tử số':measurement['complete_references'],
        'Mẫu số':measurement['reference_denominator'],'Tỷ lệ (%)':100*measurement['complete_references']/measurement['reference_denominator']}])
table([{'Phạm vi':'Các bản cumulative của phép đo','References đã đọc':measurement['references'],
        'Paper có kết quả':measurement['retrieved'],'References / paper có kết quả':measurement['mean_references']}])